# Bibliotecas

## Sistema (os, makedirs, glob)

In [ ]:
from os import makedirs
import os
import glob
from os import listdir

import dotenv
def load_dotenv():
    """Load the .env file normally and also from the current directory."""
    dotenv.load_dotenv()
    dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True))

load_dotenv()

## Básicos (numpy, math, display, locale, time, random, re)

In [2]:
# !python -m pip install jupyter


# !python -m pip install IPython
from IPython.display import display

import math

# !python -m pip install numpy
import numpy as np

import locale
# locale.setlocale(locale.LC_ALL, "pt_BR.UTF-8")  # Use "" for auto, or force e.g. to "en_US.UTF-8"

import time
from datetime import datetime, timedelta, date

from pandas.tseries.offsets import BDay # para os dias úteis
# today = datetime.datetime.today()
# print(today - BDay(4)) # 4 dias úteis atrás

# # !python -m pip install random
import random
random.seed(42)

# !python -m pip install regex
import re

## Leitura e análise de dados (Excel, Pandas, Spark)

In [3]:
# !python -m pip install findspark

# !python -m pip install openpyxl
# import openpyxl

# !python -m pip install xlsxwriter
# import xlsxwriter

# !python -m pip install xlrd
# import xlrd

# !python -m pip install python-calamine
# import python_calamine


# !python -m pip install pandas
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
# pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Finanças (yfinance, mplfinance)

In [4]:
# https://pypi.org/project/yfinance/
# https://github.com/ranaroussi/yfinance/wiki/Ticker

!python -m pip install yfinance
import yfinance as yf

!python -m pip install mplfinance
import mplfinance as mpf

# # Em R
# # https://cran.r-project.org/web/packages/BatchGetSymbols/index.html

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


## Visualização (matplotlib, seaborn, plotly)

In [5]:
# !python -m pip install matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import ticker
from matplotlib.ticker import FormatStrFormatter, StrMethodFormatter

# !python -m pip install seaborn
import seaborn as sns

# !python -m pip install plotly
# import plotly.graph_objects as go
# import plotly.express as px
# from plotly.subplots import make_subplots

# !python -m pip install graphviz
# import graphviz

# Funções

### Agrupar cada coluna (_agrupamento_cada_coluna_)

In [6]:
def agrupamento_cada_coluna(
    bd, 
    coluna_completa, 
    colunas_ignoradas = [], 
    ascending = False,
    imprime_tabelas = True,
    ):
    
    from IPython.display import display

    campos_com_erro = []
    dict_campos = {}

    colunas = bd.columns.drop(coluna_completa)

    if len(colunas_ignoradas) > 0:
        colunas = colunas.drop(colunas_ignoradas)
    
    for coluna in colunas:
        try: 
            temp_coluna = bd.fillna("(vazio)").groupby(coluna).count()[[coluna_completa]].rename(columns = {coluna_completa: "Quantidade"}).sort_values("Quantidade", ascending = ascending)

            if ascending == False:
                temp_coluna["%"] = temp_coluna["Quantidade"]/temp_coluna["Quantidade"].sum()
                temp_coluna["% acumulado"] = temp_coluna["%"].cumsum()
            
            if imprime_tabelas == True:
                display(temp_coluna)
                
            dict_campos[coluna] = temp_coluna
            # limpa(temp_coluna)
        
        except:
            campos_com_erro.append(coluna)
            #print("Campo " + coluna + " deu erro =/")
    
    return [campos_com_erro, dict_campos]

### Elimina colunas NA (_elimina_colunas_NA_)

In [7]:
def elimina_colunas_NA(
    base,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    remove_da_base = True,
    retorna_parciais = False
    ):

    tamanho_da_base = len(base)

    colunas_vazias = []
    colunas_completas = []
    colunas_parciais = []

    for coluna in base.columns:
        tamanho_da_coluna = len(base[base[coluna].isna()])
        if tamanho_da_coluna == tamanho_da_base:
            # print(coluna)
            colunas_vazias.append(coluna)
        elif tamanho_da_coluna == 0:
            colunas_completas.append(coluna)
        else:
            colunas_parciais.append(coluna)

    if imprime_colunas_vazias == True:
        if len(colunas_vazias) == 0:
            print("Não há colunas NA")
        else:
            print("Colunas vazias: " + str(colunas_vazias))
            
    if imprime_colunas_completas == True:
        if len(colunas_completas) == 0:
            print("Não há colunas completas")
        else:
            print("Colunas completas: " + str(colunas_completas))
    
    if imprime_colunas_parciais == True:
        if len(colunas_parciais) == 0:
            print("Não há colunas parciais")
        else:
            print("Colunas parciais: " + str(colunas_parciais))

    if remove_da_base == True:
        base = base.drop(colunas_vazias, axis = 1)
    
    if retorna_parciais == True:
        return [base, colunas_parciais]
    else:
        return base

### Análise exploratória básica de todos os campos da base - MUITO ÚTIL (_analise_exploratoria_)

In [8]:
def analise_exploratoria(
    bd,
    imprime_todas_colunas = False,
    imprime_info_colunas = True,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    remove_da_base = True,
    retorna_parciais = False,
    # detalhar_colunas_parciais = True,
    colunas_ignoradas = []
    ):

    # COMEÇANDO PELAS COLUNAS DISPONÍVEIS E INFO
    if imprime_todas_colunas == True:
        display(bd.columns)
    
    if imprime_info_colunas == True:
        for i in range(int(np.ceil(len(bd.columns)/20))):
            display(bd.iloc[:, (i*20):min((i+1)*20, len(bd.columns))].info())

    # DETALHAMENTO DE QUAIS COLUNAS SÃO NA OU PARCIAIS
    if retorna_parciais == True:
        [bd_semNA, colunas_parciais] = elimina_colunas_NA(
            bd,
            imprime_colunas_vazias = imprime_colunas_vazias,
            imprime_colunas_completas = imprime_colunas_completas,
            imprime_colunas_parciais = imprime_colunas_parciais,
            remove_da_base = remove_da_base,
            retorna_parciais = True
        )

        print(colunas_parciais)

        # if detalhar_colunas_parciais == True:
            # for coluna in colunas_parciais:
                # print(coluna)
                # print("# " + coluna + ": " + str(len(colunas_parciais[colunas_parciais[coluna].isna()])))
    else:
        colunas_parciais = []

        bd_semNA = elimina_colunas_NA(
            bd,
            imprime_colunas_vazias = imprime_colunas_vazias,
            imprime_colunas_completas = imprime_colunas_completas,
            imprime_colunas_parciais = imprime_colunas_parciais,
            remove_da_base = remove_da_base,
            retorna_parciais = retorna_parciais
        )

    [campos_com_erro, colunas_agrupadas] = agrupamento_cada_coluna(
        bd.reset_index(), 
        coluna_completa = bd_semNA.drop(colunas_parciais, axis = 1).reset_index().columns[0],
        colunas_ignoradas = colunas_ignoradas
    )
    
    if retorna_parciais == True:
        return [bd_semNA, colunas_parciais, colunas_agrupadas, campos_com_erro]
    else:
        return [bd_semNA, colunas_agrupadas, campos_com_erro]

## Treinamento de modelos (_treina_e_roda_modelo_)

In [9]:
def treina_e_roda_modelo(
    dados,
    colunas_treino,
    coluna_resultado,
    
    estimador = "SVC",
    proporcao = 0.25,
    SEED = 42,
    print_tamanho = True,
    print_score = True,
    print_confusion_matrix = False,
    
    DummyClassifier__estrategia = None,
    
    DecisionTreeClassifier__retornar_visualizacao = False,
    DecisionTreeClassifier__max_depth = None,
    
    RandomForestClassifier__n_estimators = 100,
    
    feature_selection = None,
    scaler = None,
):
    from sklearn.model_selection import train_test_split
    # from sklearn.preprocessing import StandardScaler
    
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    # from sklearn.svm import LinearSVC
    from sklearn.svm import SVC
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.dummy import DummyClassifier
    
    # from sklearn.metrics import accuracy_score
    from sklearn.metrics import confusion_matrix
    from sklearn.tree import export_graphviz
    import graphviz
    import os
    os.environ["PATH"] += os.pathsep + 'C:/Program Files/Graphviz/bin' # https://stackoverflow.com/questions/35064304/runtimeerror-make-sure-the-graphviz-executables-are-on-your-systems-path-aft

    np.random.seed(SEED)


    # DIVIDE A BASE NAS PORÇÕES DE TESTE E TREINO, X E Y
    [original_x_treino, original_x_teste, y_treino, y_teste] = train_test_split(
        dados.loc[:, colunas_treino],
        dados.loc[:, coluna_resultado],
        test_size = proporcao,
        # stratify = dados.loc[:, coluna_resultado] # mesma proporção de y
    )
    
    
    # SELECIONA AS FEATURES COM BASE NO ARGUMENTO E REESCALA SE NECESSÁRIO
    if feature_selection is not None:
        feature_selection.fit(original_x_treino, y_treino)
        x_treino = feature_selection.transform(original_x_treino)
        x_teste = feature_selection.transform(original_x_teste)
        
    if (scaler is None) | ((estimador is not None) * (estimador != "DecisionTreeClassifier")): # Não é necessário reescalar para árvore de decisão
        if feature_selection is None:
            x_treino = original_x_treino
            x_teste = original_x_teste
    else:
        if feature_selection is None:
            scaler.fit(original_x_treino)
            x_treino = scaler.transform(original_x_treino)
            x_teste = scaler.transform(original_x_teste)
        else:
            scaler.fit(x_treino)
            x_treino = scaler.transform(x_treino)
            x_teste = scaler.transform(x_teste)


    # IMPRIME O TAMANHO DE CADA PORÇÃO
    if print_tamanho == True:
        print("Tamanho treino: " + str(len(x_treino)))
        print("Tamanho teste: " + str(len(x_teste)))

    

    # INSTANCIA O ESTIMADOR ESCOLHIDO. SE NÃO HOUVER ESTIMADOR, RETORNA APENAS AS PORÇÕES SELECIONADAS E REESCALADAS
    if estimador is None:
        return [
            [original_x_treino, original_x_teste, y_treino, y_teste], 
            [x_treino, x_teste],
            feature_selection
        ]

    elif estimador == "SVC":
        modelo = SVC(gamma = "auto")
    
    elif estimador == "RandomForestClassifier":
        # print(estimador)
        modelo = RandomForestClassifier(n_estimators = RandomForestClassifier__n_estimators)
        print(modelo)

    elif estimador == "DecisionTreeClassifier":
        if DecisionTreeClassifier__max_depth is None:
            modelo = DecisionTreeClassifier()
        else:
            modelo = DecisionTreeClassifier(max_depth = DecisionTreeClassifier__max_depth)

    elif estimador == "MultinomialNB":
        modelo = MultinomialNB()
        
    elif estimador == "Dummy":
        if DummyClassifier__estrategia is None:  
            modelo = DummyClassifier()
        else:
            modelo = DummyClassifier(strategy = DummyClassifier__estrategia)
        
    else:
        return print("Estimador " + estimador + " não encontrado")
    
    
    # TREINA O ESTIMADOR ESCOLHIDO
    try:
        modelo.fit(x_treino, y_treino)
        print(modelo)
    except ValueError:
        return [print("Não foi possível treinar o modelo escolhido"), modelo]

    # AVALIA O MODELO
    
    # previsoes = modelo.predict(x_teste)
    # taxa_de_acerto = accuracy_score(y_teste, previsoes)

    taxa_de_acerto = modelo.score(x_teste, y_teste)
    
    if print_score == True:
        print("Taxa de acerto do modelo " + estimador + ": {:.2%}".format(taxa_de_acerto))
        
    if print_confusion_matrix == True:
        matriz_confusao = confusion_matrix(y_teste, modelo.predict(x_teste))
        # print(matriz_confusao)
        sns.set(font_scale = 2)
        sns.heatmap(matriz_confusao, annot = True, fmt = "d").set(xlabel = "Predição", ylabel = "Real")
        plt.show()
    
    # RETORNA VISÃO GRÁFICA PARA O DECISION TREE CLASSIFIER
    if (DecisionTreeClassifier__retornar_visualizacao == True) * (estimador == "DecisionTreeClassifier"):
        # return export_graphviz(modelo, out_file = None)
        dot_data = export_graphviz(
            modelo, 
            feature_names = colunas_treino,
            filled = True,
            rounded = True,
            class_names = ["Não", "Sim"]
        )
        grafico = graphviz.Source(dot_data)
        return grafico
    
    # RETORNA BASES E MODELO PARA OUTROS ESTIMADORES
    return [
        modelo, 
        [original_x_treino, original_x_teste, y_treino, y_teste], 
        [x_treino, x_teste], 
        feature_selection,
        # previsoes, 
        taxa_de_acerto
    ]

# Leitura dos dados

In [ ]:
var_caminho = os.environ.get('var_caminho_fonte') + r"\Bases"
#f

# var_arquivo = r"\Bases\Lista de ações Análise 2025-03-16.xlsx"

lista_arquivos = listdir(var_caminho)
lista_arquivos_analises = []
for arquivo in lista_arquivos:
    if arquivo.find(" Análise") > 0:
        lista_arquivos_analises.append(datetime.strptime(arquivo.split(" Análise ")[1].split(".xls")[0], "%Y-%m-%d"))

var_arquivo_mais_recente = "/Lista de ações Análise " + max(lista_arquivos_analises).strftime("%Y-%m-%d") + ".xlsx"
# print(var_arquivo_mais_recente)

bd_dados_completos = pd.read_excel(var_caminho + var_arquivo_mais_recente).set_index("Ticker")
bd_dados_completos

,Nome da Empresa,Volume no último dia útil (lido em 19/04/2025),industry,sector,fullTimeEmployees,dividendRate,dividendYield,exDividendDate,payoutRatio,beta,...,2025-04-17; Low,2025-04-17; Close,2025-04-17; Volume,2025-04-17; Dividends,2025-04-17; Stock Splits,2025-04-17; HLC,Alfa HLC; últimos 13 dias,Alfa HLC; últimos 55 dias,Martelos,Tipos de Martelos
Ticker,,,,,,,,,,,,,,,,,,,,,
COGN3,Cogna,111753600,Education & Training Services,Consumer Defensive,"24,187.00",0.26,10.70,"1,745,884,800.00",0.00,0.64,...,2.15,2.45,111753600,0.00,0,2.36,0.03,0.02,"2025-02-27, 2025-02-28, 2025-03-06, 2025-03-17...","Descida, Descida, Subida, Subida, Subida, Subi..."
PETR4,Petrobras,65752700,Oil & Gas Integrated,Energy,"41,778.00",7.09,22.98,"1,744,848,000.00",2.01,0.49,...,30.46,30.85,65752700,0.71,0,30.83,-0.52,-0.11,"2025-02-27, 2025-03-10, 2025-03-12, 2025-03-18...","Subida, Subida, Descida, Subida, Subida, Subid..."
HAPV3,Hapvida,57331500,Insurance - Life,Financial Services,NaN,NaN,NaN,"1,640,649,600.00",0.00,0.59,...,2.15,2.16,57331500,0.00,0,2.18,0.00,0.00,"2025-02-27, 2025-02-28, 2025-03-11, 2025-03-13...","Subida, Subida, Subida, Subida, Subida, Subida..."
ABEV3,Ambev,56058800,Beverages - Brewers,Consumer Defensive,"43,000.00",0.80,5.68,"1,742,169,600.00",0.73,0.36,...,13.79,14.00,56058800,0.00,0,13.96,0.02,0.05,"2025-02-26, 2025-03-18, 2025-04-04, 2025-04-07...","Descida, Subida, Descida, Descida, Subida, Des..."
MGLU3,Magazine Luiza,26837300,Specialty Retail,Consumer Cyclical,NaN,0.31,2.99,"1,745,798,400.00",0.00,1.49,...,10.16,10.20,26856800,0.00,0,10.28,-0.05,0.10,"2025-02-27, 2025-02-28, 2025-03-06, 2025-03-11...","Descida, Descida, Subida, Subida, Descida, Des..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TASA3,Taurus,26300,Aerospace & Defense,Industrials,NaN,0.20,2.24,"1,745,971,200.00",0.50,1.19,...,9.00,9.12,26300,0.00,0,9.14,0.05,0.02,"2025-02-25, 2025-02-28, 2025-03-05, 2025-03-06...","Subida, Descida, Descida, Subida, Subida, Subi..."
LOGN3,Log-In,24500,Marine Shipping,Industrials,NaN,NaN,NaN,"1,237,334,400.00",0.00,0.47,...,21.19,21.25,24500,0.00,0,21.28,0.02,-0.01,"2025-03-06, 2025-03-07, 2025-03-20, 2025-03-31...","Subida, Subida, Descida, Descida, Descida, Sub..."
BIOM3,Biomm,23800,Biotechnology,Healthcare,NaN,NaN,NaN,NaN,0.00,0.67,...,9.75,9.75,23800,0.00,0,9.93,-0.01,-0.02,"2025-02-27, 2025-03-11, 2025-03-13, 2025-03-19...","Subida, Subida, Subida, Subida, Descida, Desci..."


# Análise Exploratória

## Confirmar se coluna Ticker está duplicada

In [56]:
# bd_dados_completos[[item for item in bd_dados_completos.columns.to_list() if "Ticker" in item]]

display(len(bd_dados_completos))
display(len(bd_dados_completos.index.unique()))

246

246

## Remover outras colunas que possam se repetir (uma para cada dia)

In [12]:
lista_ignorados_datas = ["Ticker" , "HLC" , "Stock Splits" , "Dividends" , "Volume" , "Close" , "Low" , "High" , "Open"]
# lista_ignorados_datas[0]

lista_colunas_ignoradas_datas = []
for ignorado_datas in lista_ignorados_datas:
    lista_colunas_ignoradas_datas.append([item for item in bd_dados_completos.columns.to_list() if ignorado_datas in item])
# lista_colunas_ignoradas_datas

In [13]:
## Essas colunas já foram removidas no arquivo 'receber_lista_atualizada_tickers.py'

# lista_ignorados = ["industryKey", "industryDisp", "sectorKey", "volume",
#                    "sectorDisp", "regularMarketVolume", "SandP52WeekChange"]

# # lista_colunas_ignoradas_datas = []
# for ignorado_datas in lista_ignorados_datas:
#     lista_ignorados = lista_ignorados + [item for item in bd_dados_completos.columns.to_list() if ignorado_datas in item]
# print(lista_ignorados)


[bd_semNA, colunas_parciais, colunas_agrupadas, campos_com_erro] = analise_exploratoria(
    # bd_dados_completos.drop(lista_ignorados, axis = 1),
    bd_dados_completos,
    imprime_todas_colunas = False,
    # imprime_info_colunas = True,
    imprime_info_colunas = False,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    # remove_da_base = True,
    retorna_parciais = True,
    # detalhar_colunas_parciais = True,
    colunas_ignoradas = []
    )

['fullTimeEmployees', 'dividendRate', 'dividendYield', 'exDividendDate', 'payoutRatio', 'beta', 'trailingPE', 'forwardPE', 'priceToSalesTrailing12Months', 'profitMargins', 'forwardEps', 'lastSplitFactor', 'lastSplitDate', 'enterpriseToRevenue', 'enterpriseToEbitda', 'lastDividendValue', 'lastDividendDate', 'recommendationMean', 'numberOfAnalystOpinions', 'totalCash', 'totalCashPerShare', 'ebitda', 'totalDebt', 'quickRatio', 'currentRatio', 'totalRevenue', 'debtToEquity', 'revenuePerShare', 'returnOnAssets', 'returnOnEquity', 'grossProfits', 'freeCashflow', 'operatingCashflow', 'earningsGrowth', 'revenueGrowth', 'grossMargins', 'ebitdaMargins', 'epsForward', 'epsCurrentYear', 'priceEpsCurrentYear']


,Quantidade,%,% acumulado
Nome da Empresa,,,
Banco Santander,3,0.01,0.01
Taesa,3,0.01,0.02
Sanepar,3,0.01,0.04
Banco Bradesco,2,0.01,0.04
Azevedo & Travassos,2,0.01,0.05
...,...,...,...
Wilson Sons,1,0.00,0.98
Wiz Soluções,1,0.00,0.99
YDUQS,1,0.00,0.99


,Quantidade,%,% acumulado
Volume no último dia útil (lido em 19/04/2025),,,
96800,2,0.01,0.01
21300,1,0.00,0.01
23800,1,0.00,0.02
24500,1,0.00,0.02
20200,1,0.00,0.02
...,...,...,...
26837300,1,0.00,0.98
56058800,1,0.00,0.99
57331500,1,0.00,0.99


,Quantidade,%,% acumulado
industry,,,
Banks - Regional,13,0.05,0.05
Real Estate Services,12,0.05,0.10
Utilities - Renewable,11,0.04,0.15
Real Estate - Development,10,0.04,0.19
Utilities - Regulated Electric,9,0.04,0.22
...,...,...,...
Personal Services,1,0.00,0.98
Specialty Chemicals,1,0.00,0.99
Utilities - Independent Power Producers,1,0.00,0.99


,Quantidade,%,% acumulado
sector,,,
Industrials,48,0.20,0.20
Consumer Cyclical,38,0.15,0.35
Utilities,29,0.12,0.47
Financial Services,23,0.09,0.56
Real Estate,23,0.09,0.65
Basic Materials,22,0.09,0.74
Consumer Defensive,19,0.08,0.82
Healthcare,17,0.07,0.89
Communication Services,10,0.04,0.93


...

,Quantidade,%,% acumulado
2025-04-17; HLC,,,
3.73,2,0.01,0.01
5.31,2,0.01,0.02
27.50,2,0.01,0.02
0.61,1,0.00,0.03
0.72,1,0.00,0.03
...,...,...,...
52.92,1,0.00,0.98
53.26,1,0.00,0.99
62.98,1,0.00,0.99


,Quantidade,%,% acumulado
Alfa HLC; últimos 13 dias,,,
-0.66,1,0.00,0.00
-0.59,1,0.00,0.01
-0.52,1,0.00,0.01
-0.43,1,0.00,0.02
-0.39,1,0.00,0.02
...,...,...,...
0.24,1,0.00,0.98
0.37,1,0.00,0.99
0.38,1,0.00,0.99


,Quantidade,%,% acumulado
Alfa HLC; últimos 55 dias,,,
-0.26,1,0.00,0.00
-0.24,1,0.00,0.01
-0.16,1,0.00,0.01
-0.14,1,0.00,0.02
-0.13,1,0.00,0.02
...,...,...,...
0.20,1,0.00,0.98
0.34,1,0.00,0.99
0.35,1,0.00,0.99


,Quantidade,%,% acumulado
Martelos,,,
"2025-02-24, 2025-02-25, 2025-02-26, 2025-02-27, 2025-02-28, 2025-03-05, 2025-03-06, 2025-03-07, 2025-03-10, 2025-03-11, 2025-03-13, 2025-03-14, 2025-03-17, 2025-03-18, 2025-03-21, 2025-03-24, 2025-03-26, 2025-04-01, 2025-04-03, 2025-04-07, 2025-04-08, 2025-04-09, 2025-04-10, 2025-04-11, 2025-04-14, 2025-04-17,",1,0.00,0.00
"2025-02-24, 2025-02-25, 2025-02-26, 2025-02-28, 2025-03-06, 2025-03-10, 2025-03-14, 2025-03-18, 2025-03-28, 2025-04-01, 2025-04-04, 2025-04-07, 2025-04-08, 2025-04-14, 2025-04-15,",1,0.00,0.01
"2025-02-24, 2025-02-25, 2025-02-28, 2025-03-10, 2025-03-11, 2025-03-19, 2025-03-21, 2025-03-24, 2025-03-25, 2025-04-07, 2025-04-11,",1,0.00,0.01
"2025-02-24, 2025-02-25, 2025-03-05, 2025-03-20,",1,0.00,0.02
"2025-02-24, 2025-02-25, 2025-03-06, 2025-03-11, 2025-04-14,",1,0.00,0.02
...,...,...,...
"2025-03-25, 2025-03-26, 2025-04-02, 2025-04-07, 2025-04-11,",1,0.00,0.98
"2025-03-26, 2025-04-02, 2025-04-03, 2025-04-07, 2025-04-16,",1,0.00,0.99
"2025-03-27, 2025-04-04,",1,0.00,0.99


,Quantidade,%,% acumulado
Tipos de Martelos,,,
"Descida, Descida, Subida, Subida, Descida, Subida, Subida,",3,0.01,0.01
"Descida, Subida, Subida, Subida,",3,0.01,0.02
"Descida, Descida, Subida, Subida, Subida, Subida, Subida,",2,0.01,0.03
"Subida, Descida,",2,0.01,0.04
"Descida, Descida, Subida, Subida, Descida, Subida,",2,0.01,0.05
...,...,...,...
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Descida, Subida,",1,0.00,0.98
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida,",1,0.00,0.99
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida,",1,0.00,0.99


## Teste de variáveis específicas

In [18]:
display(len(bd_semNA))
display(len(bd_dados_completos))

246

246

In [19]:
colunas_parciais

['fullTimeEmployees',
 'dividendRate',
 'dividendYield',
 'exDividendDate',
 'payoutRatio',
 'beta',
 'trailingPE',
 'forwardPE',
 'priceToSalesTrailing12Months',
 'profitMargins',
 'forwardEps',
 'lastSplitFactor',
 'lastSplitDate',
 'enterpriseToRevenue',
 'enterpriseToEbitda',
 'lastDividendValue',
 'lastDividendDate',
 'recommendationMean',
 'numberOfAnalystOpinions',
 'totalCash',
 'totalCashPerShare',
 'ebitda',
 'totalDebt',
 'quickRatio',
 'currentRatio',
 'totalRevenue',
 'debtToEquity',
 'revenuePerShare',
 'returnOnAssets',
 'returnOnEquity',
 'grossProfits',
 'freeCashflow',
 'operatingCashflow',
 'earningsGrowth',
 'revenueGrowth',
 'grossMargins',
 'ebitdaMargins',
 'epsForward',
 'epsCurrentYear',
 'priceEpsCurrentYear']

In [20]:
# colunas_agrupadas
colunas_parciais[0]

# colunas_agrupadas.get("52WeekChange")

colunas_agrupadas.get(colunas_parciais[0])

,Quantidade,%,% acumulado
fullTimeEmployees,,,
(vazio),142,0.58,0.58
"6,047.00",3,0.01,0.59
854.00,3,0.01,0.60
"55,646.00",3,0.01,0.61
"6,000.00",2,0.01,0.62
...,...,...,...
"64,616.00",1,0.00,0.98
"87,000.00",1,0.00,0.99
"100,000.00",1,0.00,0.99


In [21]:
campos_com_erro

[]

### _recommendationKey_

In [62]:
display(bd_dados_completos[bd_dados_completos["recommendationKey"] == 'buy'].sort_index().index)
display(bd_dados_completos[bd_dados_completos["recommendationKey"] == 'strong_buy'].sort_index().index)

Index(['AGRO3', 'ALOS3', 'ALUP11', 'ANIM3', 'ARML3', 'ASAI3', 'B3SA3', 'BBAS3',
       'BBSE3', 'BLAU3', 'BMGB4', 'BMOB3', 'BPAC11', 'BPAN4', 'BRAP4', 'BRAV3',
       'BRFS3', 'BRKM5', 'CAML3', 'CBAV3', 'CCRO3', 'COGN3', 'CSAN3', 'CSED3',
       'CVCB3', 'CXSE3', 'CYRE3', 'DXCO3', 'ECOR3', 'ELMD3', 'ENGI11', 'FIQE3',
       'FRAS3', 'GGBR4', 'GMAT3', 'GOAU4', 'GUAR3', 'HAPV3', 'HBSA3', 'HYPE3',
       'INTB3', 'IRBR3', 'KLBN11', 'LJQQ3', 'LOGG3', 'LREN3', 'LWSA3', 'MATD3',
       'MOVI3', 'MRFG3', 'MRVE3', 'MULT3', 'MYPK3', 'NEOE3', 'NTCO3', 'PNVL3',
       'PSSA3', 'RADL3', 'RAIL3', 'RAPT4', 'RDOR3', 'RECV3', 'SBFG3', 'SBSP3',
       'SEER3', 'SLCE3', 'SMTO3', 'STBP3', 'SUZB3', 'SYNE3', 'TEND3', 'TFCO4',
       'TIMS3', 'TOTS3', 'TTEN3', 'VALE3', 'VAMO3', 'VBBR3', 'VIVT3', 'VTRU3',
       'WEGE3', 'YDUQ3', 'ZAMP3'],
      dtype='object', name='Ticker')

Index(['AZZA3', 'CEAB3', 'CPLE3', 'CPLE6', 'CURY3', 'DIRR3', 'ELET3', 'EQTL3',
       'GGPS3', 'ITSA4', 'ITUB4', 'JALL3', 'JBSS3', 'JHSF3', 'LAVV3', 'MDNE3',
       'MELK3', 'ORVR3', 'PETR3', 'PETR4', 'PLPL3', 'POMO4', 'PRIO3', 'PRNR3',
       'RAIZ4', 'RANI3', 'RENT3', 'SMFT3', 'SOJA3', 'SRNA3', 'VIVA3', 'VULC3'],
      dtype='object', name='Ticker')

# Preparação de dados

## Remover todas as colunas parciais

In [130]:
bd_dados_completos_sem_colunas_parciais = bd_dados_completos.drop(colunas_parciais, axis = 1).copy()
bd_dados_completos_sem_colunas_parciais.columns

Index(['Nome da Empresa', 'Volume no último dia útil (lido em 19/04/2025)',
       'industry', 'sector', 'averageVolume', 'averageVolume10days',
       'averageDailyVolume10Day', 'marketCap', 'fiftyDayAverage',
       'twoHundredDayAverage',
       ...
       '2025-04-17; Low', '2025-04-17; Close', '2025-04-17; Volume',
       '2025-04-17; Dividends', '2025-04-17; Stock Splits', '2025-04-17; HLC',
       'Alfa HLC; últimos 13 dias', 'Alfa HLC; últimos 55 dias', 'Martelos',
       'Tipos de Martelos'],
      dtype='object', length=321)

## One hot encoding

In [131]:
## VER OS TIPOS DE TODAS AS COLUNAS

# var_tamanho_print = 22

# for var_contador in range(round(len(bd_dados_completos_sem_colunas_parciais.columns)/22)):
#     # display((var_contador*var_tamanho_print + 1))
#     # display(((var_contador+1)*var_tamanho_print))
#     display(bd_dados_completos_sem_colunas_parciais.iloc[:, (var_contador*var_tamanho_print + 1):((var_contador+1)*var_tamanho_print)].info())

# bd_dados_completos_sem_colunas_parciais.select_dtypes("object")#.columns

In [132]:
for var_coluna in bd_dados_completos_sem_colunas_parciais.select_dtypes("object").columns:
    display(colunas_agrupadas.get(var_coluna))

,Quantidade,%,% acumulado
Nome da Empresa,,,
Banco Santander,3,0.01,0.01
Taesa,3,0.01,0.02
Sanepar,3,0.01,0.04
Banco Bradesco,2,0.01,0.04
Azevedo & Travassos,2,0.01,0.05
...,...,...,...
Wilson Sons,1,0.00,0.98
Wiz Soluções,1,0.00,0.99
YDUQS,1,0.00,0.99


,Quantidade,%,% acumulado
industry,,,
Banks - Regional,13,0.05,0.05
Real Estate Services,12,0.05,0.10
Utilities - Renewable,11,0.04,0.15
Real Estate - Development,10,0.04,0.19
Utilities - Regulated Electric,9,0.04,0.22
...,...,...,...
Personal Services,1,0.00,0.98
Specialty Chemicals,1,0.00,0.99
Utilities - Independent Power Producers,1,0.00,0.99


,Quantidade,%,% acumulado
sector,,,
Industrials,48,0.20,0.20
Consumer Cyclical,38,0.15,0.35
Utilities,29,0.12,0.47
Financial Services,23,0.09,0.56
Real Estate,23,0.09,0.65
Basic Materials,22,0.09,0.74
Consumer Defensive,19,0.08,0.82
Healthcare,17,0.07,0.89
Communication Services,10,0.04,0.93


,Quantidade,%,% acumulado
recommendationKey,,,
buy,83,0.34,0.34
none,83,0.34,0.67
hold,43,0.17,0.85
strong_buy,32,0.13,0.98
underperform,5,0.02,1.00


,Quantidade,%,% acumulado
Martelos,,,
"2025-02-24, 2025-02-25, 2025-02-26, 2025-02-27, 2025-02-28, 2025-03-05, 2025-03-06, 2025-03-07, 2025-03-10, 2025-03-11, 2025-03-13, 2025-03-14, 2025-03-17, 2025-03-18, 2025-03-21, 2025-03-24, 2025-03-26, 2025-04-01, 2025-04-03, 2025-04-07, 2025-04-08, 2025-04-09, 2025-04-10, 2025-04-11, 2025-04-14, 2025-04-17,",1,0.00,0.00
"2025-02-24, 2025-02-25, 2025-02-26, 2025-02-28, 2025-03-06, 2025-03-10, 2025-03-14, 2025-03-18, 2025-03-28, 2025-04-01, 2025-04-04, 2025-04-07, 2025-04-08, 2025-04-14, 2025-04-15,",1,0.00,0.01
"2025-02-24, 2025-02-25, 2025-02-28, 2025-03-10, 2025-03-11, 2025-03-19, 2025-03-21, 2025-03-24, 2025-03-25, 2025-04-07, 2025-04-11,",1,0.00,0.01
"2025-02-24, 2025-02-25, 2025-03-05, 2025-03-20,",1,0.00,0.02
"2025-02-24, 2025-02-25, 2025-03-06, 2025-03-11, 2025-04-14,",1,0.00,0.02
...,...,...,...
"2025-03-25, 2025-03-26, 2025-04-02, 2025-04-07, 2025-04-11,",1,0.00,0.98
"2025-03-26, 2025-04-02, 2025-04-03, 2025-04-07, 2025-04-16,",1,0.00,0.99
"2025-03-27, 2025-04-04,",1,0.00,0.99


,Quantidade,%,% acumulado
Tipos de Martelos,,,
"Descida, Descida, Subida, Subida, Descida, Subida, Subida,",3,0.01,0.01
"Descida, Subida, Subida, Subida,",3,0.01,0.02
"Descida, Descida, Subida, Subida, Subida, Subida, Subida,",2,0.01,0.03
"Subida, Descida,",2,0.01,0.04
"Descida, Descida, Subida, Subida, Descida, Subida,",2,0.01,0.05
...,...,...,...
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Descida, Subida,",1,0.00,0.98
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida,",1,0.00,0.99
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida,",1,0.00,0.99


In [133]:
lista_remover_colunas_object = bd_dados_completos_sem_colunas_parciais.select_dtypes("object").columns
lista_remover_colunas_object

bd_dados_completos_limpar_object = bd_dados_completos_sem_colunas_parciais.drop(lista_remover_colunas_object, axis = 1).copy()

bd_dados_completos_limpar_object

,Volume no último dia útil (lido em 19/04/2025),averageVolume,averageVolume10days,averageDailyVolume10Day,marketCap,fiftyDayAverage,twoHundredDayAverage,trailingAnnualDividendRate,trailingAnnualDividendYield,trailingEps,...,2025-04-17; Open,2025-04-17; High,2025-04-17; Low,2025-04-17; Close,2025-04-17; Volume,2025-04-17; Dividends,2025-04-17; Stock Splits,2025-04-17; HLC,Alfa HLC; últimos 13 dias,Alfa HLC; últimos 55 dias
Ticker,,,,,,,,,,,,,,,,,,,,,
COGN3,111753600,54038432,74166360,74166360,4430286336,1.85,1.48,0.07,0.03,0.46,...,2.26,2.48,2.15,2.45,111753600,0.00,0,2.36,0.03,0.02
PETR4,65752700,37900975,73125810,73125810,414883151872,35.65,36.94,5.73,0.19,3.40,...,30.67,31.18,30.46,30.85,65752700,0.71,0,30.83,-0.52,-0.11
HAPV3,57331500,100621951,62252560,62252560,16026746880,2.23,3.13,0.00,0.00,-0.09,...,2.22,2.22,2.15,2.16,57331500,0.00,0,2.18,0.00,0.00
ABEV3,56058800,35657248,42149260,42149260,219060797440,12.70,12.50,0.67,0.05,0.91,...,13.90,14.10,13.79,14.00,56058800,0.00,0,13.96,0.02,0.05
MGLU3,26837300,29897058,36728390,36728390,7513472512,8.91,9.49,0.00,0.00,0.61,...,10.41,10.48,10.16,10.20,26856800,0.00,0,10.28,-0.05,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TASA3,26300,9633,14800,14800,1151837696,8.26,9.71,0.20,0.02,0.61,...,9.01,9.29,9.00,9.12,26300,0.00,0,9.14,0.05,0.02
LOGN3,24500,40854,19440,19440,2254370048,21.35,27.04,0.00,0.00,1.41,...,21.24,21.39,21.19,21.25,24500,0.00,0,21.28,0.02,-0.01
BIOM3,23800,66737,16610,16610,1236709504,9.85,9.54,0.00,0.00,-0.63,...,10.00,10.30,9.75,9.75,23800,0.00,0,9.93,-0.01,-0.02


## Balanceamento

In [134]:
# # # SÓ CONSEGUI FAZER FUNCIONAR REMOVENDO O PARÂMETRO stratify

# from sklearn.model_selection import train_test_split
# dados = bd_dados_completos_limpar_object
# colunas_treino = bd_dados_completos_limpar_object.drop(var_coluna_objetivo, axis = 1).columns
# coluna_resultado = var_coluna_objetivo
# proporcao = 0.10

# # display(dados.loc[:, colunas_treino])
# # list(dados.loc[:, coluna_resultado].values)
# # dados.loc[:, [coluna_resultado]].iloc[:, 0]

# [original_x_treino, original_x_teste, y_treino, y_teste] = train_test_split(
#     dados.loc[:, colunas_treino],
#     dados.loc[:, [coluna_resultado]],
#     test_size = proporcao,
#     # stratify = dados.loc[:, [coluna_resultado]] # mesma proporção de y
# )

# # # [original_x_treino, original_x_teste, y_treino, y_teste]
# display(len(original_x_treino))
# display(len(original_x_teste))
# display(len(y_treino))
# display(len(y_teste))

## Normalização

## Removendo todas as variáveis relacionadas a dados dos últimos 13 dias

In [202]:
lista_variaveis_diarias = [
    "; Open",
    "; High",
    "; Low",
    "; Close",
    "; Volume",
    "; Dividends",
    "; Stock Splits",
    "; HLC"]
lista_colunas_variaveis_diarias_remover = ["Alfa HLC; últimos 55 dias"]
# lista_colunas_variaveis_diarias_remover = []

for var_coluna in bd_dados_completos_limpar_object.columns:
    for var_diaria in lista_variaveis_diarias:
        if (var_diaria in var_coluna):
            if (datetime.strptime(var_coluna.split("; ")[0], "%Y-%m-%d") > (datetime.today() - timedelta(days=13))):
                # display(var_coluna)
                lista_colunas_variaveis_diarias_remover.append(var_coluna)


# display(lista_colunas_variaveis_diarias_remover)
bd_dados_completos_limpar_colunas_diarias = bd_dados_completos_limpar_object.drop(lista_colunas_variaveis_diarias_remover, axis = 1).copy()
# bd_dados_completos_limpar_colunas_diarias

bd_dados_completos_limpar_colunas_diarias.iloc[:5, -20:]

,2025-04-02; Dividends,2025-04-02; Stock Splits,2025-04-02; HLC,2025-04-03; Open,2025-04-03; High,2025-04-03; Low,2025-04-03; Close,2025-04-03; Volume,2025-04-03; Dividends,2025-04-03; Stock Splits,2025-04-03; HLC,2025-04-04; Open,2025-04-04; High,2025-04-04; Low,2025-04-04; Close,2025-04-04; Volume,2025-04-04; Dividends,2025-04-04; Stock Splits,2025-04-04; HLC,Alfa HLC; últimos 13 dias
Ticker,,,,,,,,,,,,,,,,,,,,
COGN3,0.00,0,2.11,2.07,2.22,2.07,2.18,77999600,0.00,0,2.16,2.11,2.11,2.01,2.06,63292200,0.00,0,2.06,0.03
PETR4,0.00,0,36.26,35.32,35.47,34.79,35.18,65896000,0.00,0,35.15,33.94,34.21,33.04,33.76,87263200,0.00,0,33.67,-0.52
HAPV3,0.00,0,2.21,2.17,2.29,2.17,2.25,62135700,0.00,0,2.24,2.20,2.21,2.11,2.13,67506100,0.00,0,2.15,0.00
ABEV3,0.00,0,13.60,13.64,14.14,13.61,13.85,44926800,0.00,0,13.87,13.73,13.90,13.56,13.71,59559600,0.00,0,13.72,0.02
MGLU3,0.00,0,10.92,11.16,11.85,11.14,11.80,32976900,0.00,0,11.60,11.34,11.49,10.74,10.83,40210300,0.00,0,11.02,-0.05


# [MODELO 1] Decision Tree Classifier

## Definição da variável objetivo

In [193]:
## SE A VARIÁVEL OBJETIVO FOR CATEGÓRICA, PODEMOS USAR SE O PREÇO AUMENTOU, SUBIU OU FICOU ESTÁVEL

bd_dados_analises = bd_dados_completos_limpar_colunas_diarias.copy()

# var_margem_estavel = 0.02
var_margem_estavel = 0
var_coluna_objetivo = "Resultado"

bd_dados_analises.loc[bd_dados_analises["Alfa HLC; últimos 13 dias"] > var_margem_estavel, var_coluna_objetivo] = "Subiu"
bd_dados_analises.loc[bd_dados_analises["Alfa HLC; últimos 13 dias"] < -1*var_margem_estavel, var_coluna_objetivo] = "Desceu"
bd_dados_analises.loc[
    (bd_dados_analises["Alfa HLC; últimos 13 dias"] <= var_margem_estavel) * (bd_dados_analises["Alfa HLC; últimos 13 dias"] >= -1*var_margem_estavel), 
    var_coluna_objetivo] = "Estável"

display(bd_dados_analises[["Alfa HLC; últimos 13 dias", "Resultado"]].head(20))
bd_dados_analises = bd_dados_analises.drop(["Alfa HLC; últimos 13 dias"], axis = 1)

display(bd_dados_analises["Resultado"].value_counts())

,Alfa HLC; últimos 13 dias,Resultado
Ticker,,
COGN3,0.03,Subiu
PETR4,-0.52,Desceu
HAPV3,0.00,Subiu
ABEV3,0.02,Subiu
MGLU3,-0.05,Desceu
LWSA3,0.05,Subiu
ITSA4,0.05,Subiu
BBDC4,0.03,Subiu
ANIM3,0.04,Subiu


Resultado
Subiu     134
Desceu    112
Name: count, dtype: int64

## Treinamento e score

In [194]:
var_modelo_treinado_DecisionTreeClassifier = treina_e_roda_modelo(
    dados = bd_dados_analises,
    colunas_treino = bd_dados_analises.drop(var_coluna_objetivo, axis = 1).columns,
    coluna_resultado = var_coluna_objetivo,
    
    estimador = "DecisionTreeClassifier",
    proporcao = 0.10,
    SEED = 42,
    print_tamanho = True,
    print_score = True,
    print_confusion_matrix = False,
    
    DummyClassifier__estrategia = None,
    
    DecisionTreeClassifier__retornar_visualizacao = False,
    DecisionTreeClassifier__max_depth = None,
    
    RandomForestClassifier__n_estimators = 100,
    
    feature_selection = None,
    scaler = None,
)

# type(modelo)
# modelo.fit(x_treino, y_treino)

Tamanho treino: 221
Tamanho teste: 25
DecisionTreeClassifier()
Taxa de acerto do modelo DecisionTreeClassifier: 76.00%


## Análise do modelo

In [195]:
# [
#     modelo, 
#     [original_x_treino, original_x_teste, y_treino, y_teste], 
#     [x_treino, x_teste], 
#     feature_selection,
#     taxa_de_acerto
# ]

tabela_resultado = pd.DataFrame(var_modelo_treinado_DecisionTreeClassifier[1][3])
tabela_resultado = tabela_resultado.rename(columns = {"Resultado": "Resultado esperado"})

tabela_resultado["Resultado gerado"] = list(var_modelo_treinado_DecisionTreeClassifier[0].predict(
    var_modelo_treinado_DecisionTreeClassifier[1][1]))

sum(tabela_resultado["Resultado gerado"] == tabela_resultado["Resultado esperado"])/len(tabela_resultado)

# tabela_resultado

0.76

# [MODELO 2] Dummy

## Treinamento e score

In [196]:
var_modelo_treinado_Dummy = treina_e_roda_modelo(
    dados = bd_dados_analises,
    colunas_treino = bd_dados_analises.drop(var_coluna_objetivo, axis = 1).columns,
    coluna_resultado = var_coluna_objetivo,
    
    estimador = "Dummy",
    proporcao = 0.10,
    SEED = 42,
    print_tamanho = True,
    print_score = True,
    print_confusion_matrix = False,
    
    DummyClassifier__estrategia = None,
    
    DecisionTreeClassifier__retornar_visualizacao = False,
    DecisionTreeClassifier__max_depth = None,
    
    RandomForestClassifier__n_estimators = 100,
    
    feature_selection = None,
    scaler = None,
)

# type(modelo)
# modelo.fit(x_treino, y_treino)

Tamanho treino: 221
Tamanho teste: 25
DummyClassifier()
Taxa de acerto do modelo Dummy: 52.00%


## Análise do modelo

In [197]:
# [
#     modelo, 
#     [original_x_treino, original_x_teste, y_treino, y_teste], 
#     [x_treino, x_teste], 
#     feature_selection,
#     taxa_de_acerto
# ]

tabela_resultado = pd.DataFrame(var_modelo_treinado_Dummy[1][3])
tabela_resultado = tabela_resultado.rename(columns = {"Resultado": "Resultado esperado"})

tabela_resultado["Resultado gerado"] = list(var_modelo_treinado_Dummy[0].predict(
    var_modelo_treinado_Dummy[1][1]))

# sum(tabela_resultado["Resultado gerado"] == tabela_resultado["Resultado esperado"])/len(tabela_resultado)

tabela_resultado

,Resultado esperado,Resultado gerado
Ticker,,
POMO4,Subiu,Subiu
ITSA4,Subiu,Subiu
AGXY3,Desceu,Subiu
OIBR3,Desceu,Subiu
BLAU3,Desceu,Subiu
KLBN3,Desceu,Subiu
WIZC3,Desceu,Subiu
AZZA3,Subiu,Subiu
PCAR3,Subiu,Subiu


# [MODELO 3] RandomForestClassifier

## Treinamento e score

In [198]:
var_modelo_treinado_RandomForestClassifier = treina_e_roda_modelo(
    dados = bd_dados_analises,
    colunas_treino = bd_dados_analises.drop(var_coluna_objetivo, axis = 1).columns,
    coluna_resultado = var_coluna_objetivo,
    
    estimador = "RandomForestClassifier",
    proporcao = 0.10,
    SEED = 42,
    print_tamanho = True,
    print_score = True,
    print_confusion_matrix = False,
    
    DummyClassifier__estrategia = None,
    
    DecisionTreeClassifier__retornar_visualizacao = False,
    DecisionTreeClassifier__max_depth = None,
    
    RandomForestClassifier__n_estimators = 100,
    
    feature_selection = None,
    scaler = None,
)

# type(modelo)
# modelo.fit(x_treino, y_treino)

Tamanho treino: 221
Tamanho teste: 25
RandomForestClassifier()
RandomForestClassifier()
Taxa de acerto do modelo RandomForestClassifier: 72.00%


## Análise do modelo

In [199]:
# [
#     modelo, 
#     [original_x_treino, original_x_teste, y_treino, y_teste], 
#     [x_treino, x_teste], 
#     feature_selection,
#     taxa_de_acerto
# ]

tabela_resultado = pd.DataFrame(var_modelo_treinado_RandomForestClassifier[1][3])
tabela_resultado = tabela_resultado.rename(columns = {"Resultado": "Resultado esperado"})

tabela_resultado["Resultado gerado"] = list(var_modelo_treinado_RandomForestClassifier[0].predict(
    var_modelo_treinado_RandomForestClassifier[1][1]))

# sum(tabela_resultado["Resultado gerado"] == tabela_resultado["Resultado esperado"])/len(tabela_resultado)

tabela_resultado

,Resultado esperado,Resultado gerado
Ticker,,
POMO4,Subiu,Desceu
ITSA4,Subiu,Subiu
AGXY3,Desceu,Desceu
OIBR3,Desceu,Desceu
BLAU3,Desceu,Desceu
KLBN3,Desceu,Desceu
WIZC3,Desceu,Desceu
AZZA3,Subiu,Subiu
PCAR3,Subiu,Subiu


# Modelo com regressão

In [ ]:
# # SE A VARIÁVEL OBJETIVO FOR CONTÍNUA, PODEMOS USAR O PREÇO, COMO NESSE CASO

# var_coluna_objetivo = bd_dados_completos_limpar_object[[item for item in bd_dados_completos_limpar_object.columns.to_list() if "; Close" in item]]

# var_coluna_objetivo = var_coluna_objetivo.columns[-1]

# var_coluna_objetivo